# Week 1 — Data Exploration & Cleaning
**Project:** RAG Ticket Intelligence System

**Goal:** Understand the ITSM dataset, clean it, and prepare it for embedding in Week 2.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
sns.set_theme(style='whitegrid')

print('Libraries loaded.')

## 2. Load Data
Place your downloaded CSV in `data/raw/` and update the path below.

In [ ]:
DATA_PATH = '../data/raw/itsm_tickets.csv'  # update filename if different

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

## 3. Basic Info

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 4. Missing Values

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})[missing > 0]

## 5. Category Distribution

In [ ]:
# Update column name to match your dataset
CATEGORY_COL = 'category'  # e.g. 'Category', 'ticket_type', etc.

if CATEGORY_COL in df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    df[CATEGORY_COL].value_counts().plot(kind='bar', ax=ax)
    ax.set_title('Ticket Category Distribution')
    ax.set_xlabel('Category')
    ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print(f'Column "{CATEGORY_COL}" not found. Available columns:', df.columns.tolist())

## 6. Severity Distribution

In [ ]:
SEVERITY_COL = 'priority'  # e.g. 'severity', 'impact', 'Priority'

if SEVERITY_COL in df.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    df[SEVERITY_COL].value_counts().plot(kind='bar', ax=ax, color='coral')
    ax.set_title('Severity / Priority Distribution')
    ax.set_xlabel('Priority')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print(f'Column "{SEVERITY_COL}" not found. Available columns:', df.columns.tolist())

## 7. Text Length Analysis

In [ ]:
TEXT_COL = 'description'  # the main text field — update if different

if TEXT_COL in df.columns:
    df['text_length'] = df[TEXT_COL].astype(str).apply(len)
    print(df['text_length'].describe())

    fig, ax = plt.subplots(figsize=(9, 4))
    df['text_length'].hist(bins=50, ax=ax, color='steelblue')
    ax.set_title('Ticket Description Length Distribution')
    ax.set_xlabel('Character Count')
    ax.set_ylabel('Frequency')
    plt.tight_layout()
    plt.show()
else:
    print(f'Column "{TEXT_COL}" not found. Available columns:', df.columns.tolist())

## 8. Data Cleaning

In [ ]:
import re

def clean_text(text):
    if pd.isnull(text):
        return ''
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)          # collapse whitespace
    text = re.sub(r'[^\w\s.,!?-]', '', text)  # remove special chars
    return text

df_clean = df.copy()

# Clean text columns
for col in [TEXT_COL]:  # add more text cols here if needed
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(clean_text)

# Drop rows where the main text is empty
before = len(df_clean)
df_clean = df_clean[df_clean[TEXT_COL].str.len() > 10]
print(f'Dropped {before - len(df_clean)} rows with empty/short descriptions.')

# Drop duplicates
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=[TEXT_COL])
print(f'Dropped {before - len(df_clean)} duplicate rows.')

print(f'Final dataset shape: {df_clean.shape}')
df_clean.head()

## 9. Save Cleaned Data

In [ ]:
OUTPUT_PATH = '../data/processed/tickets_clean.csv'
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f'Saved cleaned data to {OUTPUT_PATH}')

## 10. Summary
- Total tickets (raw): `{len(df)}`
- Total tickets (clean): `{len(df_clean)}`
- Key columns identified: update this after exploring
- Next step: Week 2 — generate embeddings and load into ChromaDB